# Qwen 2.5 14B — Few-Shot v3 (Background-vs-Focus filter)

Tercera iteración del prompt fewshot. Mantiene el algoritmo deductivo de v2 pero añade:

1. **Step 1 con validación de foco** — distingue *background context* vs *research focus* para evitar que el modelo extraiga métricas que solo aparecen como contexto retórico.
2. **Criterio operativo de "actually analyzed"** — una métrica solo cuenta si el abstract reporta un número, comparación, salida de modelo o metodología de medición sobre ella.
3. **Re-verify en Step 3** — segunda pasada explícita sobre estudios sociales/policy/teóricos para desanclar la decisión del Step 1.

**Hipótesis:** baja el sesgo positivista (None→PB1 = 10 en v2) sin romper el Hit@K (72.7% en v2).

In [1]:
import pandas as pd
import requests
import json
import time
import os
from pathlib import Path

## 1. Carga de datos y construcción de reglas PB

In [2]:
ROOT_DIR = Path.cwd().resolve().parents[2]
LLM_DIR = Path.cwd().resolve().parents[0]
OUTPUTS_DIR = LLM_DIR / 'outputs' / 'inferences'
GROUND_TRUTH_DIR = LLM_DIR / 'outputs' / 'ground_truth'
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

ruta_corpus = ROOT_DIR / 'data' / 'corpus' / 'master_corpus_mixto_1000_clean_enriched.csv'
ruta_pbs = ROOT_DIR / 'corpus_PB' / 'data' / 'pb_reference.csv'

df_corpus = pd.read_csv(ruta_corpus)
df_pbs = pd.read_csv(ruta_pbs)

df_human = pd.read_csv(GROUND_TRUTH_DIR / 'validacion_real.csv', sep=';', encoding='utf-8')
ids_validados = df_human['doc_id'].dropna().astype(str).unique().tolist()
df_sample = df_corpus[df_corpus['doc_id'].isin(ids_validados)].copy()
print(f'Papers a evaluar: {len(df_sample)}')

Papers a evaluar: 149


In [3]:
pb_rules = ''
for _, row in df_pbs.iterrows():
    pb_rules += f"- PB Code: {row['pb_code']} ({row['pb_name']})\n"
    pb_rules += f"  * Core Definition: {row['short_definition']}\n"
    pb_rules += f"  * UPV Context: Look for terms like: {row['applied_keywords_upv']}\n"
    pb_rules += f"  * ACTIVATION LOGIC: {row['activation_logic']}\n"
    pb_rules += f"  * EXCLUSION RULE (CRITICAL): {row['exclusion_notes']}\n\n"
print(pb_rules[:600], '...')

- PB Code: PB1 (Climate Change)
  * Core Definition: Human-driven alteration of the climate system through greenhouse gas emissions and associated warming.
  * UPV Context: Look for terms like: renewable energy systems; photovoltaic systems; wind energy; smart grids; power systems; energy efficiency; industrial decarbonization; low-carbon materials; carbon capture and storage; sustainable mobility; zero-emission vehicles; life cycle carbon assessment
  * ACTIVATION LOGIC: Core and explicit climate terms can activate PB1. Applied UPV terms only reinforce when they co-occur with emissions, carbo ...


## 2. Configuración LLM + prompt v3

In [4]:
OLLAMA_URL = 'http://localhost:11434/api/generate'
MODEL_NAME = 'qwen2.5:14b'

def classify_abstract_v3(abstract_text):
    prompt = f"""You are a strict scientific evaluator analyzing research abstracts from the Universitat Politècnica de València (UPV) against the Planetary Boundaries (PBs) framework.

Your task: identify the Planetary Boundaries that the research actually MEASURES or MODELS.

### PLANETARY BOUNDARIES RULES:
{pb_rules}

### YOUR ANALYTICAL ALGORITHM (mandatory deductive checklist)

Execute these steps internally, in order, ON THE REAL TEXT BELOW. Do not invent metrics that are not in the abstract. Do not reuse rhetoric from these instructions.

Step 1 — BIOPHYSICAL EXTRACTION & VALIDATION
   List the physical, chemical, biological, or ecological metrics that this specific research ACTUALLY MEASURES, MODELS, or ANALYZES. Quote them.
   - CRITICAL DISTINCTION: Differentiate between \"background context\" and \"research focus\". If the abstract merely mentions \"climate change\", \"warming\", or \"emissions\" as the background reason for a purely economic, social, legal, or governance study, IT DOES NOT COUNT. The study itself must generate or analyze biophysical/environmental data.
   - OPERATIONAL CRITERION: A metric counts as \"actually analyzed\" only if the abstract reports a number, a comparison, a model output, or a measurement methodology around it. A bare mention without quantification, modelling or methodological treatment is background.
   - If the list of *actually analyzed* biophysical metrics is EMPTY, the verdict is \"None\" and you skip to Step 5.

Step 2 — CANDIDATE MAPPING
   For EACH extracted metric, map it to the PB whose Core Definition / Activation Logic it triggers. A paper can map to several PBs.
   - Biosphere/ecological responses (species, biodiversity, ecosystems) -> PB7 is a legitimate PRIMARY candidate.
   - Particulate matter, PM2.5, PM10, aerosols, AOD -> PB9 (atmospheric aerosol loading), NOT PB1.
   - \"Climate\" or \"warming\" alone is NOT enough for PB1: require an explicit climate driver or ecological impact.
   - Nutrients (N, P, fertilizers) -> PB4. Water quantity/quality -> PB5. Pick the one that is the MATHEMATICAL focus.

Step 3 — EXCLUSION CHECK
   For every candidate PB from Step 2, apply its EXCLUSION RULE. Discard any PB whose exclusion rule is explicitly violated.
   - Re-verify: Is the paper purely social science, policy, or theoretical computing? If yes, and the biophysical terms were just background context, discard the PB to prevent positivity bias.

Step 4 — HIERARCHY
   Among the surviving PBs, select as `primary_pb` the dominant analytical focus. Any other surviving PB goes to `secondary_pbs`. PBs discarded in Step 3 go to `rejected_pbs`.

Step 5 — OUTPUT
   Write `reasoning_process` as a concise factual trace of Steps 1-4 referring to the actual text. Do NOT echo phrases from these instructions.

### ABSTRACT TO EVALUATE:
<text>
{abstract_text}
</text>

### OUTPUT FORMAT
Output ONLY valid JSON matching this exact schema. Do not include markdown blocks or extra text outside the JSON:

{{
    \"reasoning_process\": \"Your concise factual trace of Steps 1-4 quoting the abstract.\",
    \"primary_pb\": {{\"pb_code\": \"PBX\", \"confidence\": \"High/Medium/Low\"}} or null,
    \"secondary_pbs\": [{{\"pb_code\": \"PBY\", \"confidence\": \"High/Medium/Low\"}}],
    \"rejected_pbs\": [\"PBZ\", \"PBW\"]
}}
"""

    payload = {
        'model': MODEL_NAME,
        'prompt': prompt,
        'format': 'json',
        'stream': False,
        'options': {'temperature': 0.0, 'top_p': 0.9},
    }
    start_time = time.time()
    try:
        response = requests.post(OLLAMA_URL, json=payload)
        response.raise_for_status()
        return response.json().get('response', '{}'), time.time() - start_time
    except Exception as e:
        return json.dumps({'error': str(e)}), time.time() - start_time

## 3. Parser de la respuesta JSON

In [5]:
def parse_llm_output(raw_text):
    try:
        start_idx = raw_text.find('{')
        end_idx = raw_text.rfind('}')
        if start_idx == -1 or end_idx == -1:
            raise ValueError('No JSON structure found.')
        data = json.loads(raw_text[start_idx:end_idx + 1])

        reasoning = data.get('reasoning_process', '')

        prim_pb = data.get('primary_pb')
        if isinstance(prim_pb, dict):
            prim_code = prim_pb.get('pb_code', 'None')
            prim_conf = prim_pb.get('confidence', 'Unknown')
        else:
            prim_code, prim_conf = 'None', 'N/A'

        sec_pbs = data.get('secondary_pbs', []) or []
        sec_codes = ', '.join([item.get('pb_code', '') for item in sec_pbs if isinstance(item, dict)]) or 'None'

        rej_pbs = data.get('rejected_pbs', []) or []
        rej_codes = ', '.join(rej_pbs) if rej_pbs else 'None'

        return reasoning, prim_code, prim_conf, sec_codes, rej_codes
    except Exception as e:
        return f'Error: {e}', 'Error', 'Error', 'Error', 'Error'

## 4. Bucle de evaluación

In [6]:
output_filename = OUTPUTS_DIR / 'qwen2.5_14b_fewshot_v3.csv'
if os.path.exists(output_filename):
    os.remove(output_filename)

total_papers = len(df_sample)
print(f'Iniciando evaluación FEWSHOT v3 con modelo: {MODEL_NAME}')
print('-' * 70)

for i, (_, row) in enumerate(df_sample.iterrows(), start=1):
    print(f"[{i}/{total_papers}] Doc: {row['doc_id']}...", end=' ', flush=True)
    llm_out, t_elapsed = classify_abstract_v3(row['clean_abstract'])
    reasoning, p_code, p_conf, s_codes, r_codes = parse_llm_output(llm_out)
    print(f'[{t_elapsed:.1f}s] Pri: {p_code} | Sec: {s_codes} | Rej: {r_codes}')

    df_fila = pd.DataFrame([{
        'doc_id': row['doc_id'],
        'llm_primary_pb': p_code,
        'llm_primary_conf': p_conf,
        'llm_secondary_pbs': s_codes,
        'llm_rejected_pbs': r_codes,
        'llm_reasoning': reasoning,
        'inference_time_sec': t_elapsed,
    }])
    if not os.path.isfile(output_filename):
        df_fila.to_csv(output_filename, index=False)
    else:
        df_fila.to_csv(output_filename, mode='a', header=False, index=False)

print('=' * 70)
print(f'Finalizado. CSV guardado en: {output_filename}')

Iniciando evaluación FEWSHOT v3 con modelo: qwen2.5:14b
----------------------------------------------------------------------
[1/149] Doc: b39624d6c38a... [6.3s] Pri: None | Sec: None | Rej: PB1
[2/149] Doc: 8581e74341ad... [7.0s] Pri: PB9 | Sec: None | Rej: PB1, PB2, PB3, PB4, PB5, PB6, PB7, PB8
[3/149] Doc: 03c55a425a95... [6.4s] Pri: PB1 | Sec: None | Rej: PB2, PB3, PB4, PB5, PB6, PB7, PB8, PB9
[4/149] Doc: 8d8ab7ed834f... [5.9s] Pri: PB1 | Sec: None | Rej: PB4, PB5, PB6, PB7, PB8, PB9
[5/149] Doc: ff6fb1e2be19... [5.0s] Pri: None | Sec: None | Rej: PB1
[6/149] Doc: 44cb9ddc20a6... [6.1s] Pri: PB1 | Sec: None | Rej: PB2, PB4, PB5, PB6, PB7, PB8, PB9
[7/149] Doc: 5194c7c1714e... [4.3s] Pri: None | Sec: None | Rej: PB1
[8/149] Doc: ff153d55d5fd... [4.0s] Pri: None | Sec: None | Rej: None
[9/149] Doc: a3d6daa1a396... [3.5s] Pri: None | Sec: None | Rej: None
[10/149] Doc: eaf6d52d5457... [5.8s] Pri: None | Sec: None | Rej: PB1, PB9
[11/149] Doc: 11e601dd40d6... [6.9s] Pri: PB1 | Sec: N